In [4]:
import argparse
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from scipy import stats
from sklearn.calibration import calibration_curve
from sklearn.metrics import roc_auc_score, brier_score_loss
from statsbombpy import sb
import xgboost as xgb
import statsmodels.api as sm

warnings.filterwarnings("ignore")

COMP_ID = 11  # La Liga, per StatsBomb open data
DATA_DIR = Path("data")
OUT_DIR = Path("poster_outputs")
DATA_DIR.mkdir(exist_ok=True)
OUT_DIR.mkdir(exist_ok=True)

PASS_TYPE_FLAGS = {
    "Cross": "pass_cross_f",
    "Switch": "pass_switch_f",
    "Through ball": "pass_through_ball_f",
    "Cut-back": "pass_cut_back_f",
}

# Set this to True in the notebook to also run the bonus 2020/21
# event-only vs 360 comparison at the end (Cell 11).
RUN_360_BONUS = False

# Optionally limit to a handful of seasons while iterating (None = all seasons)
SEASON_FILTER = None   # e.g. ["2020/2021", "2019/2020", "2018/2019"]


In [5]:
def get_laliga_seasons():
    """Return a DataFrame of every (competition_id, season_id, season_name)
    for men's La Liga available in StatsBomb's open data."""
    comps = sb.competitions()
    laliga = comps[
        (comps["competition_id"] == COMP_ID) & (comps["competition_gender"] == "male")
    ].sort_values("season_name")
    return laliga[["competition_id", "season_id", "season_name"]].reset_index(drop=True)


seasons = get_laliga_seasons()
if SEASON_FILTER:
    seasons = seasons[seasons["season_name"].isin(SEASON_FILTER)]

print(f"Found {len(seasons)} La Liga season(s) to process:")
seasons


Found 18 La Liga season(s) to process:


,competition_id,season_id,season_name
0,11,278,1973/1974
1,11,37,2004/2005
2,11,38,2005/2006
3,11,39,2006/2007
4,11,40,2007/2008
5,11,41,2008/2009
6,11,21,2009/2010
7,11,22,2010/2011
8,11,23,2011/2012
9,11,24,2012/2013


In [8]:
cache_path = DATA_DIR / f"passes_278.parquet"
df_1974 = pd.read_parquet(cache_path)
print(df_1974['pass_body_part'].value_counts(dropna=False))

ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - `Import pyarrow` failed. pyarrow is required for parquet support. Use pip or conda to install the pyarrow package.
 - `Import fastparquet` failed. fastparquet is required for parquet support. Use pip or conda to install the fastparquet package.